# NSE 5× Turnaround — V3 Causal Portfolio Backtest

Corrected V3 research notebook.

### Execution model
- Signal is generated using information available at the **close of T**.
- Entry executes at the **open of T+1**.
- Every exit decision is also made using information available through **close of T** and executes at **open of T+1**.
- No hard initial stop.
- Trailing-stop sensitivity: **20%, 30%, 40%, 50%, 60%, 70%**.
- Nifty regime filters: no filter, Nifty > 200DMA, Nifty 50DMA > 200DMA, and bullish regime = Nifty > 200DMA + 50DMA > 200DMA + positive 20D return.
- Portfolio metrics, trade diagnostics, chronological walk-forward validation, and a dataset freshness hard-fail are included.

This notebook is a research framework. It does not automatically eliminate survivorship bias, corporate-action errors, historical-universe bias, or market-impact constraints.


## 1. Configuration


In [ ]:
from pathlib import Path
import json
import warnings
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/quant/data/parquet")
RESULTS_DIR = DATA_DIR.parent / "results" / "5x_turnaround_v3"

# Optional explicit Nifty benchmark file.
NIFTY_PATH = None

# Hard freshness guard.
# Set this lower only when intentionally running a historical snapshot.
EXPECTED_MIN_END_DATE = pd.Timestamp("2026-09-01")

STARTING_CAPITAL = 1_000_000.0
MAX_POSITIONS = 20
MAX_ALLOCATION_PER_POSITION = 1.0 / MAX_POSITIONS

# Configurable research assumptions.
SLIPPAGE_BPS = 5.0
BUY_COST_BPS = 5.0
SELL_COST_BPS = 15.0

ENTRY_SCORE_MIN = 8
TRAILING_STOPS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

FAILURE_EXIT_DAYS = 252
MAX_HOLD_DAYS = 756

WF_TRAIN_YEARS = 5
WF_VALID_YEARS = 2
WF_TEST_YEARS = 2

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Expected minimum stock date:", EXPECTED_MIN_END_DATE.date())


## 2. Data helpers


In [ ]:
try:
    import pyarrow.parquet as pq
except ImportError:
    pq = None

def list_parquet_files(root: Path) -> List[Path]:
    return sorted(root.rglob("*.parquet"))

def normalize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename = {}
    for c in df.columns:
        lc = str(c).lower().strip()
        if lc in {"date", "timestamp", "datetime", "trade_date"}:
            rename[c] = "date"
        elif lc in {"symbol", "ticker", "tradingsymbol", "security"}:
            rename[c] = "symbol"
        elif lc in {"open", "open_price"}:
            rename[c] = "open"
        elif lc in {"high", "high_price"}:
            rename[c] = "high"
        elif lc in {"low", "low_price"}:
            rename[c] = "low"
        elif lc in {"close", "close_price", "last"}:
            rename[c] = "close"
        elif lc in {"volume", "vol"}:
            rename[c] = "volume"

    df = df.rename(columns=rename)

    required = {"date", "open", "high", "low", "close"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required OHLC columns: {sorted(missing)}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    for c in ["open", "high", "low", "close", "volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["date", "open", "high", "low", "close"])

    if "symbol" not in df.columns:
        df["symbol"] = "UNKNOWN"

    return (
        df.sort_values(["symbol", "date"])
          .drop_duplicates(["symbol", "date"], keep="last")
          .reset_index(drop=True)
    )

files = list_parquet_files(DATA_DIR)
print(f"Parquet files found: {len(files):,}")
if not files:
    raise FileNotFoundError(f"No Parquet files found below {DATA_DIR}")


## 3. Hard dataset freshness guard


In [ ]:
def parquet_date_bounds(path: Path) -> Tuple[Optional[pd.Timestamp], Optional[pd.Timestamp]]:
    try:
        if pq is not None:
            pf = pq.ParquetFile(path)
            names = [x.lower() for x in pf.schema.names]
            date_col = None
            for candidate in ["date", "trade_date", "timestamp", "datetime"]:
                if candidate in names:
                    date_col = pf.schema.names[names.index(candidate)]
                    break

            if date_col:
                mins, maxs = [], []
                idx = names.index(date_col.lower())
                for rg in pf.metadata.row_groups:
                    st = rg.column(idx).statistics
                    if st and st.has_min_max:
                        mins.append(pd.to_datetime(st.min))
                        maxs.append(pd.to_datetime(st.max))
                if mins and maxs:
                    return min(mins), max(maxs)
    except Exception:
        pass

    try:
        df = pd.read_parquet(path, columns=["date"])
        s = pd.to_datetime(df["date"], errors="coerce").dropna()
        if len(s):
            return s.min(), s.max()
    except Exception:
        return None, None

    return None, None

bounds = []
for p in files:
    mn, mx = parquet_date_bounds(p)
    if mx is not None:
        bounds.append((p, mn, mx))

if not bounds:
    raise RuntimeError("Could not determine Parquet date bounds.")

actual_min = min(x[1] for x in bounds if x[1] is not None)
actual_max = max(x[2] for x in bounds)

print("Dataset minimum:", actual_min.date())
print("Dataset maximum:", actual_max.date())
print("Required minimum:", EXPECTED_MIN_END_DATE.date())

if actual_max < EXPECTED_MIN_END_DATE:
    raise RuntimeError(
        f"STALE DATASET FAILURE: latest stock date is {actual_max.date()}, "
        f"below required {EXPECTED_MIN_END_DATE.date()}. "
        "Run the incremental NSE downloader first."
    )

print("✓ Freshness guard passed.")


## 4. Load stock data


In [ ]:
frames = []

for p in files:
    try:
        x = pd.read_parquet(p)
        x = normalize_ohlcv(x)
        if len(x):
            frames.append(x)
    except Exception as e:
        print(f"Skipping {p.name}: {e}")

if not frames:
    raise RuntimeError("No readable Parquet data.")

stocks = normalize_ohlcv(pd.concat(frames, ignore_index=True))
stocks["symbol"] = stocks["symbol"].astype(str).str.upper().str.strip()
stocks = stocks.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Rows: {len(stocks):,}")
print(f"Symbols: {stocks['symbol'].nunique():,}")
print(f"Range: {stocks['date'].min().date()} → {stocks['date'].max().date()}")

if stocks["date"].max() < EXPECTED_MIN_END_DATE:
    raise RuntimeError(
        f"STALE DATASET FAILURE after load: {stocks['date'].max().date()} "
        f"< {EXPECTED_MIN_END_DATE.date()}"
    )


## 5. Nifty benchmark and regime filters


In [ ]:
def load_nifty(path: Optional[Path] = None) -> pd.DataFrame:
    candidates = []
    if path:
        candidates.append(Path(path))

    candidates += [
        DATA_DIR / "nifty50.parquet",
        DATA_DIR / "NIFTY50.parquet",
        DATA_DIR / "nifty.parquet",
        DATA_DIR / "NIFTY.parquet",
        DATA_DIR / "NIFTY_50.parquet",
    ]

    for p in candidates:
        if p.exists():
            x = normalize_ohlcv(pd.read_parquet(p))
            return x[["date","open","high","low","close"]].sort_values("date")

    names = {"NIFTY", "NIFTY50", "NIFTY 50", "^NSEI"}
    x = stocks[stocks["symbol"].isin(names)].copy()

    if not x.empty:
        return (
            x.groupby("date", as_index=False)
             .agg(open=("open","first"),
                  high=("high","max"),
                  low=("low","min"),
                  close=("close","last"))
             .sort_values("date")
        )

    return pd.DataFrame()

nifty = load_nifty(NIFTY_PATH)

REGIMES = ["none", "nifty_gt_200", "50_gt_200", "bull"]

if nifty.empty:
    print("WARNING: Nifty benchmark not found.")
    print("Only the no-filter regime can run.")
else:
    nifty["sma50"] = nifty["close"].rolling(50).mean()
    nifty["sma200"] = nifty["close"].rolling(200).mean()
    nifty["ret20"] = nifty["close"].pct_change(20)

    nifty["regime_none"] = True
    nifty["regime_nifty_gt_200"] = nifty["close"] > nifty["sma200"]
    nifty["regime_50_gt_200"] = nifty["sma50"] > nifty["sma200"]
    nifty["regime_bull"] = (
        (nifty["close"] > nifty["sma200"]) &
        (nifty["sma50"] > nifty["sma200"]) &
        (nifty["ret20"] > 0)
    )

    nifty = nifty.set_index("date").sort_index()

    print(
        "Nifty range:",
        nifty.index.min().date(),
        "→",
        nifty.index.max().date()
    )

    if nifty.index.max() < EXPECTED_MIN_END_DATE:
        raise RuntimeError(
            f"Nifty benchmark is stale: {nifty.index.max().date()} "
            f"< {EXPECTED_MIN_END_DATE.date()}"
        )


## 6. Causal technical features


In [ ]:
LOOKBACK_60 = 60
LOOKBACK_252 = 252

def add_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    g["sma20"] = g["close"].rolling(20).mean()
    g["sma50"] = g["close"].rolling(50).mean()
    g["sma100"] = g["close"].rolling(100).mean()
    g["sma200"] = g["close"].rolling(200).mean()

    g["high60_prev"] = g["high"].rolling(LOOKBACK_60).max().shift(1)
    g["high252_prev"] = g["high"].rolling(LOOKBACK_252).max().shift(1)
    g["low252_prev"] = g["low"].rolling(LOOKBACK_252).min().shift(1)

    g["ret20"] = g["close"].pct_change(20)
    g["ret60"] = g["close"].pct_change(60)
    g["ret120"] = g["close"].pct_change(120)
    g["ret252"] = g["close"].pct_change(252)

    g["vol60"] = g["close"].pct_change().rolling(60).std()

    if "volume" in g.columns:
        g["vol_avg60"] = g["volume"].rolling(60).mean()
        g["volume_ratio60"] = g["volume"] / g["vol_avg60"]
    else:
        g["volume_ratio60"] = np.nan

    g["drawdown252"] = g["close"] / g["high252_prev"] - 1.0
    g["range252"] = g["high252_prev"] / g["low252_prev"] - 1.0

    g["above20"] = g["close"] > g["sma20"]
    g["sma20_rising"] = g["sma20"] > g["sma20"].shift(5)
    g["higher_low"] = g["low"] > g["low"].shift(5)
    g["breakout60"] = g["close"] > g["high60_prev"]
    g["above50"] = g["close"] > g["sma50"]
    g["sma50_gt100"] = g["sma50"] > g["sma100"]
    g["positive_momentum"] = g["ret20"] > 0
    g["volume_expansion"] = g["volume_ratio60"] > 1.5
    g["above200"] = g["close"] > g["sma200"]

    g["score"] = (
        g["above20"].fillna(False).astype(int)
        + g["sma20_rising"].fillna(False).astype(int)
        + g["higher_low"].fillna(False).astype(int)
        + g["breakout60"].fillna(False).astype(int)
        + g["above50"].fillna(False).astype(int)
        + g["sma50_gt100"].fillna(False).astype(int)
        + g["positive_momentum"].fillna(False).astype(int)
        + g["volume_expansion"].fillna(False).astype(int)
        + g["above200"].fillna(False).astype(int)
    )

    # Preserve the V2-style final signal family.
    g["entry_signal"] = (
        (g["score"] >= ENTRY_SCORE_MIN) &
        g["breakout60"] &
        g["above50"] &
        g["positive_momentum"] &
        g["sma50_gt100"]
    )

    return g

features = (
    stocks.groupby("symbol", group_keys=False)
          .apply(add_features, include_groups=False)
          .reset_index(drop=True)
)

if nifty.empty:
    for r in REGIMES:
        features[f"regime_{r}"] = (r == "none")
else:
    regime_frame = nifty[
        ["regime_none","regime_nifty_gt_200","regime_50_gt_200","regime_bull"]
    ].copy()

    features = features.merge(
        regime_frame,
        left_on="date",
        right_index=True,
        how="left"
    )

    features["regime_none"] = features["regime_none"].fillna(False)
    features["regime_nifty_gt_200"] = features["regime_nifty_gt_200"].fillna(False)
    features["regime_50_gt_200"] = features["regime_50_gt_200"].fillna(False)
    features["regime_bull"] = features["regime_bull"].fillna(False)

print("Feature rows:", f"{len(features):,}")
print("Raw signals:", int(features["entry_signal"].sum()))


## 7. Strict next-open trade simulator

For a signal at close `T`:

- entry = open of `T+1`
- exit decision on close `D` = exit execution at open `D+1`
- trailing stop uses the highest close known through `D`
- failure exit uses close/SMA50 known through `D`
- no hard initial stop
- final-date positions are left open rather than being closed using the same day's close


In [ ]:
@dataclass
class Trade:
    symbol: str
    signal_date: pd.Timestamp
    entry_date: pd.Timestamp
    entry_price: float
    exit_date: pd.Timestamp
    exit_price: float
    exit_reason: str
    gross_return: float
    net_return: float
    mae: float
    mfe: float
    hold_days: int
    score: float

def apply_slippage(price: float, side: str) -> float:
    if side == "buy":
        return price * (1.0 + SLIPPAGE_BPS / 10000.0)
    return price * (1.0 - SLIPPAGE_BPS / 10000.0)

def net_trade_return(entry_raw: float, exit_raw: float) -> float:
    entry = apply_slippage(entry_raw, "buy") * (1.0 + BUY_COST_BPS / 10000.0)
    exit_ = apply_slippage(exit_raw, "sell") * (1.0 - SELL_COST_BPS / 10000.0)
    return exit_ / entry - 1.0

def simulate_trade(
    g: pd.DataFrame,
    signal_idx: int,
    trailing_stop: float,
) -> Optional[Trade]:
    if signal_idx + 1 >= len(g):
        return None

    entry_idx = signal_idx + 1
    entry_row = g.iloc[entry_idx]
    entry_raw = float(entry_row["open"])

    if not np.isfinite(entry_raw) or entry_raw <= 0:
        return None

    entry_price = apply_slippage(entry_raw, "buy")

    mae = 0.0
    mfe = 0.0
    last_idx = min(len(g) - 1, entry_idx + MAX_HOLD_DAYS)

    for i in range(entry_idx, last_idx + 1):
        row = g.iloc[i]
        hi = float(row["high"])
        lo = float(row["low"])

        mae = min(mae, lo / entry_raw - 1.0)
        mfe = max(mfe, hi / entry_raw - 1.0)

        # We cannot make an exit decision from today's close and execute
        # at today's open. Decision uses row i close and execution is i+1 open.
        if i >= last_idx:
            break

        current_close = float(row["close"])
        current_sma50 = (
            float(row["sma50"]) if pd.notna(row["sma50"]) else np.nan
        )

        prior_closes = g.iloc[entry_idx:i+1]["close"]
        highest_close = float(prior_closes.max())

        stop_price = highest_close * (1.0 - trailing_stop)

        reason = None

        # Trailing stop decision at close i -> execute open i+1.
        if current_close <= stop_price:
            reason = "TRAILING_STOP"

        # Failure exit decision at close i -> execute open i+1.
        elif (
            (i - entry_idx + 1) >= FAILURE_EXIT_DAYS
            and np.isfinite(current_sma50)
            and current_close < current_sma50
            and current_close < entry_price
        ):
            reason = "FAILURE_EXIT"

        # Time exit decision at close i -> execute open i+1.
        elif (i - entry_idx + 1) >= MAX_HOLD_DAYS:
            reason = "TIME_EXIT"

        if reason:
            exit_idx = i + 1
            exit_row = g.iloc[exit_idx]
            exit_raw = float(exit_row["open"])

            if not np.isfinite(exit_raw) or exit_raw <= 0:
                continue

            gross = exit_raw / entry_raw - 1.0
            net = net_trade_return(entry_raw, exit_raw)

            return Trade(
                symbol=str(g.iloc[signal_idx]["symbol"]),
                signal_date=pd.Timestamp(g.iloc[signal_idx]["date"]),
                entry_date=pd.Timestamp(entry_row["date"]),
                entry_price=float(entry_price),
                exit_date=pd.Timestamp(exit_row["date"]),
                exit_price=float(apply_slippage(exit_raw, "sell")),
                exit_reason=reason,
                gross_return=float(gross),
                net_return=float(net),
                mae=float(mae),
                mfe=float(mfe),
                hold_days=int((exit_row["date"] - entry_row["date"]).days),
                score=float(g.iloc[signal_idx]["score"]),
            )

    # No causal exit exists before the dataset ends.
    return None


## 8. Build causal trade sets


In [ ]:
def build_trade_set(
    features: pd.DataFrame,
    trailing_stop: float,
    regime: str
) -> pd.DataFrame:
    if regime not in REGIMES:
        raise ValueError(regime)

    records = []

    for symbol, g in features.groupby("symbol", sort=False):
        g = g.sort_values("date").reset_index(drop=True)
        eligible = (
            g["entry_signal"] &
            g[f"regime_{regime}"]
        )

        for idx in np.flatnonzero(eligible.to_numpy()):
            trade = simulate_trade(g, int(idx), trailing_stop)
            if trade is not None:
                records.append(trade.__dict__)

    cols = [
        "symbol","signal_date","entry_date","entry_price",
        "exit_date","exit_price","exit_reason",
        "gross_return","net_return","mae","mfe","hold_days","score"
    ]

    if not records:
        return pd.DataFrame(columns=cols)

    return (
        pd.DataFrame(records)
          .sort_values(["entry_date","symbol"])
          .reset_index(drop=True)
    )

trade_sets = {}

for regime in REGIMES:
    if nifty.empty and regime != "none":
        continue

    for stop in TRAILING_STOPS:
        print(f"Building regime={regime}, stop={stop:.0%}")
        trade_sets[(regime, stop)] = build_trade_set(
            features, stop, regime
        )
        print("  closed causal trades:", len(trade_sets[(regime, stop)]))


## 9. Portfolio engine

The engine consumes the causal trade table:

- trade entry date = next open after signal
- trade exit date = next open after exit decision
- exits are processed before new entries
- max 20 concurrent positions
- target allocation is 1 / 20 of current equity
- unused cash remains in cash
- positions are marked at close for the equity curve


In [ ]:
def simulate_portfolio(
    features: pd.DataFrame,
    trades: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    if trades.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    entry_map = {}
    exit_map = {}

    for idx, r in trades.iterrows():
        entry_map.setdefault(pd.Timestamp(r["entry_date"]), []).append((idx, r))
        exit_map.setdefault(pd.Timestamp(r["exit_date"]), []).append((idx, r))

    # Price lookup for execution/marking.
    px = (
        features.set_index(["date","symbol"])[["open","close"]]
        .sort_index()
    )

    dates = sorted(pd.to_datetime(features["date"].unique()))

    cash = STARTING_CAPITAL
    positions = {}
    closed = []
    equity_rows = []

    for date in dates:
        date = pd.Timestamp(date)

        # 1) Exit positions at today's open.
        for trade_id, r in exit_map.get(date, []):
            symbol = str(r["symbol"])
            if symbol not in positions:
                continue

            p = positions.pop(symbol)
            qty = p["qty"]
            exit_px = float(r["exit_price"])

            gross = qty * exit_px
            sell_cost = gross * SELL_COST_BPS / 10000.0
            proceeds = gross - sell_cost

            cash += proceeds

            closed.append({
                **r.to_dict(),
                "portfolio_entry_price": p["entry_price"],
                "portfolio_exit_price": exit_px,
                "quantity": qty,
                "notional_entry": p["notional"],
                "notional_exit": gross,
                "portfolio_net_return": proceeds / p["cash_deployed"] - 1.0,
            })

        # 2) Enter new trades at today's open.
        for trade_id, r in entry_map.get(date, []):
            symbol = str(r["symbol"])

            if symbol in positions or len(positions) >= MAX_POSITIONS:
                continue

            try:
                open_raw = float(px.loc[(date, symbol), "open"])
            except Exception:
                continue

            if not np.isfinite(open_raw) or open_raw <= 0:
                continue

            current_equity = cash + sum(
                p["qty"] * p["last_close"]
                for p in positions.values()
            )

            target = min(
                current_equity * MAX_ALLOCATION_PER_POSITION,
                cash / (1.0 + BUY_COST_BPS / 10000.0)
            )

            if target <= 0:
                continue

            buy_px = apply_slippage(open_raw, "buy")
            buy_cost = target * BUY_COST_BPS / 10000.0
            qty = target / buy_px

            cash -= target + buy_cost

            positions[symbol] = {
                "entry_price": buy_px,
                "qty": qty,
                "notional": target,
                "cash_deployed": target + buy_cost,
                "last_close": buy_px,
            }

        # 3) Mark positions at today's close.
        for symbol, p in positions.items():
            try:
                close_px = float(px.loc[(date, symbol), "close"])
                if np.isfinite(close_px):
                    p["last_close"] = close_px
            except Exception:
                pass

        position_value = sum(
            p["qty"] * p["last_close"]
            for p in positions.values()
        )

        equity = cash + position_value

        equity_rows.append({
            "date": date,
            "equity": equity,
            "cash": cash,
            "position_value": position_value,
            "open_positions": len(positions),
            "exposure": position_value / equity if equity > 0 else 0.0,
        })

    open_rows = [
        {
            "symbol": symbol,
            "entry_price": p["entry_price"],
            "quantity": p["qty"],
            "last_close": p["last_close"],
            "unrealized_return": p["last_close"] / p["entry_price"] - 1.0,
        }
        for symbol, p in positions.items()
    ]

    return (
        pd.DataFrame(equity_rows),
        pd.DataFrame(closed),
        pd.DataFrame(open_rows),
    )


## 10. Portfolio metrics and calendar-year returns


In [ ]:
def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    return float((equity / peak - 1.0).min())

def annualized_sharpe(equity: pd.Series) -> float:
    r = equity.pct_change().dropna()
    if len(r) < 2 or r.std() == 0:
        return np.nan
    return float(np.sqrt(252) * r.mean() / r.std())

def annualized_sortino(equity: pd.Series) -> float:
    r = equity.pct_change().dropna()
    downside = r[r < 0]
    if len(downside) < 2 or downside.std() == 0:
        return np.nan
    return float(np.sqrt(252) * r.mean() / downside.std())

def portfolio_metrics(
    equity_df: pd.DataFrame,
    closed_df: pd.DataFrame
) -> Dict:
    if equity_df.empty:
        return {}

    e = equity_df.sort_values("date").copy()
    end = float(e["equity"].iloc[-1])
    days = max((e["date"].iloc[-1] - e["date"].iloc[0]).days, 1)
    years = days / 365.25

    cagr = (end / STARTING_CAPITAL) ** (1.0 / years) - 1.0
    dd = max_drawdown(e["equity"])

    return {
        "start_date": e["date"].iloc[0],
        "end_date": e["date"].iloc[-1],
        "starting_capital": STARTING_CAPITAL,
        "ending_equity": end,
        "CAGR": cagr,
        "max_drawdown": dd,
        "Sharpe": annualized_sharpe(e["equity"]),
        "Sortino": annualized_sortino(e["equity"]),
        "Calmar": cagr / abs(dd) if dd < 0 else np.nan,
        "closed_trades": int(len(closed_df)),
        "avg_open_positions": float(e["open_positions"].mean()),
        "max_open_positions": int(e["open_positions"].max()),
        "avg_exposure": float(e["exposure"].mean()),
        "max_exposure": float(e["exposure"].max()),
    }

def calendar_year_returns(equity_df: pd.DataFrame) -> pd.DataFrame:
    e = equity_df.sort_values("date").copy()
    e["year"] = e["date"].dt.year

    year_end = (
        e.groupby("year", as_index=False)
         .tail(1)[["year","date","equity"]]
         .sort_values("year")
         .reset_index(drop=True)
    )

    rows = []
    previous = STARTING_CAPITAL

    for _, r in year_end.iterrows():
        rows.append({
            "year": int(r["year"]),
            "year_end_date": r["date"],
            "year_end_equity": r["equity"],
            "return": r["equity"] / previous - 1.0,
        })
        previous = r["equity"]

    return pd.DataFrame(rows)


## 11. Full trailing-stop × Nifty-regime matrix


In [ ]:
matrix_rows = {}
portfolio_store = {}

for (regime, stop), trades in trade_sets.items():
    print(f"Backtesting regime={regime}, trailing_stop={stop:.0%}")

    eq, closed, open_pos = simulate_portfolio(features, trades)

    if eq.empty:
        continue

    metrics = portfolio_metrics(eq, closed)
    row = {
        "regime": regime,
        "trailing_stop": stop,
        **metrics,
    }

    matrix_rows[(regime, stop)] = row
    portfolio_store[(regime, stop)] = {
        "equity": eq,
        "closed": closed,
        "open": open_pos,
        "yearly": calendar_year_returns(eq),
    }

matrix = pd.DataFrame(matrix_rows.values())

display_cols = [
    "regime","trailing_stop","ending_equity","CAGR",
    "max_drawdown","Sharpe","Sortino","Calmar",
    "closed_trades","avg_open_positions","max_open_positions",
    "avg_exposure"
]

display(
    matrix[display_cols]
    .sort_values(["regime","trailing_stop"])
    .reset_index(drop=True)
)


## 12. Export full-sample portfolio results


In [ ]:
matrix.to_csv(
    RESULTS_DIR / "v3_regime_trailing_stop_matrix.csv",
    index=False
)

for (regime, stop), bundle in portfolio_store.items():
    tag = regime.replace(">", "gt").replace(" ", "_")
    stop_tag = f"{int(stop*100):02d}"
    prefix = f"{tag}_stop_{stop_tag}"

    bundle["equity"].to_parquet(
        RESULTS_DIR / f"{prefix}_equity.parquet",
        index=False
    )
    bundle["closed"].to_parquet(
        RESULTS_DIR / f"{prefix}_closed_trades.parquet",
        index=False
    )
    bundle["open"].to_parquet(
        RESULTS_DIR / f"{prefix}_open_positions.parquet",
        index=False
    )
    bundle["yearly"].to_csv(
        RESULTS_DIR / f"{prefix}_yearly.csv",
        index=False
    )

print("Exported:", RESULTS_DIR)


## 13. Trade-level diagnostics


In [ ]:
def trade_diagnostics(trades: pd.DataFrame) -> Dict:
    if trades.empty:
        return {}

    return {
        "trades": len(trades),
        "win_rate": float((trades["net_return"] > 0).mean()),
        "mean_return": float(trades["net_return"].mean()),
        "median_return": float(trades["net_return"].median()),
        "avg_mae": float(trades["mae"].mean()),
        "avg_mfe": float(trades["mfe"].mean()),
        "2x_rate": float((trades["mfe"] >= 1.0).mean()),
        "3x_rate": float((trades["mfe"] >= 2.0).mean()),
        "5x_rate": float((trades["mfe"] >= 4.0).mean()),
        "median_hold_days": float(trades["hold_days"].median()),
    }

diag_rows = []

for (regime, stop), trades in trade_sets.items():
    d = trade_diagnostics(trades)
    if d:
        diag_rows.append({
            "regime": regime,
            "trailing_stop": stop,
            **d
        })

diagnostics = pd.DataFrame(diag_rows)

display(
    diagnostics.sort_values(["regime","trailing_stop"]).reset_index(drop=True)
)

diagnostics.to_csv(
    RESULTS_DIR / "v3_trade_diagnostics.csv",
    index=False
)


## 14. Walk-forward windows


In [ ]:
def make_walk_forward_windows(
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> List[Dict]:

    windows = []
    cursor = pd.Timestamp(start).normalize()

    while True:
        train_end = (
            cursor +
            pd.DateOffset(years=WF_TRAIN_YEARS) -
            pd.Timedelta(days=1)
        )
        valid_end = train_end + pd.DateOffset(years=WF_VALID_YEARS)
        test_end = valid_end + pd.DateOffset(years=WF_TEST_YEARS)

        if test_end > end:
            break

        windows.append({
            "train_start": cursor,
            "train_end": train_end,
            "valid_start": train_end + pd.Timedelta(days=1),
            "valid_end": valid_end,
            "test_start": valid_end + pd.Timedelta(days=1),
            "test_end": test_end,
        })

        cursor = cursor + pd.DateOffset(years=WF_TEST_YEARS)

    return windows

wf_windows = make_walk_forward_windows(
    features["date"].min(),
    features["date"].max()
)

print("Walk-forward folds:", len(wf_windows))

for i, w in enumerate(wf_windows, 1):
    print(
        i,
        "| train", w["train_start"].date(), "→", w["train_end"].date(),
        "| valid", w["valid_start"].date(), "→", w["valid_end"].date(),
        "| test", w["test_start"].date(), "→", w["test_end"].date()
    )


## 15. Walk-forward trade validation

The strategy is rule-based, so the train period is currently used as historical context rather than fitting a machine-learning model. Validation/test periods remain chronological and untouched by random shuffling.

For a later parameter-learning version, parameters selected on train/validation should be frozen before the test period.


In [ ]:
def window_trade_stats(
    trades: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp
) -> Dict:
    x = trades[
        (pd.to_datetime(trades["entry_date"]) >= start) &
        (pd.to_datetime(trades["entry_date"]) <= end) &
        (pd.to_datetime(trades["exit_date"]) <= end)
    ].copy()

    if x.empty:
        return {
            "trades": 0,
            "win_rate": np.nan,
            "mean_return": np.nan,
            "median_return": np.nan,
            "2x_rate": np.nan,
            "3x_rate": np.nan,
            "5x_rate": np.nan,
        }

    return {
        "trades": len(x),
        "win_rate": float((x["net_return"] > 0).mean()),
        "mean_return": float(x["net_return"].mean()),
        "median_return": float(x["net_return"].median()),
        "2x_rate": float((x["mfe"] >= 1.0).mean()),
        "3x_rate": float((x["mfe"] >= 2.0).mean()),
        "5x_rate": float((x["mfe"] >= 4.0).mean()),
    }

wf_rows = []

for fold, w in enumerate(wf_windows, 1):
    for regime in REGIMES:
        if nifty.empty and regime != "none":
            continue

        for stop in TRAILING_STOPS:
            trades = trade_sets[(regime, stop)]

            train = window_trade_stats(
                trades, w["train_start"], w["train_end"]
            )
            valid = window_trade_stats(
                trades, w["valid_start"], w["valid_end"]
            )
            test = window_trade_stats(
                trades, w["test_start"], w["test_end"]
            )

            wf_rows.append({
                "fold": fold,
                "regime": regime,
                "trailing_stop": stop,
                "train_start": w["train_start"],
                "train_end": w["train_end"],
                "valid_start": w["valid_start"],
                "valid_end": w["valid_end"],
                "test_start": w["test_start"],
                "test_end": w["test_end"],
                **{f"train_{k}": v for k,v in train.items()},
                **{f"valid_{k}": v for k,v in valid.items()},
                **{f"test_{k}": v for k,v in test.items()},
            })

wf = pd.DataFrame(wf_rows)

display(
    wf[
        [
            "fold","regime","trailing_stop",
            "test_start","test_end",
            "test_trades","test_win_rate",
            "test_mean_return","test_median_return",
            "test_2x_rate","test_3x_rate","test_5x_rate"
        ]
    ].sort_values(["fold","regime","trailing_stop"])
)

wf.to_csv(
    RESULTS_DIR / "v3_walk_forward_trade_validation.csv",
    index=False
)


## 16. Isolated walk-forward portfolio tests


In [ ]:
def simulate_portfolio_subset(
    features: pd.DataFrame,
    trades: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    x = trades[
        (pd.to_datetime(trades["entry_date"]) >= start) &
        (pd.to_datetime(trades["entry_date"]) <= end) &
        (pd.to_datetime(trades["exit_date"]) <= end)
    ].copy()

    if x.empty:
        return pd.DataFrame(), pd.DataFrame()

    entry_map = {}
    exit_map = {}

    for idx, r in x.iterrows():
        entry_map.setdefault(pd.Timestamp(r["entry_date"]), []).append((idx, r))
        exit_map.setdefault(pd.Timestamp(r["exit_date"]), []).append((idx, r))

    px = features.set_index(["date","symbol"])[["open","close"]].sort_index()

    all_dates = pd.to_datetime(features["date"].unique())
    dates = sorted(d for d in all_dates if start <= d <= end)

    cash = STARTING_CAPITAL
    positions = {}
    closed = []
    eq_rows = []

    for date in dates:
        date = pd.Timestamp(date)

        # Exit first.
        for trade_id, r in exit_map.get(date, []):
            symbol = str(r["symbol"])
            if symbol not in positions:
                continue

            p = positions.pop(symbol)
            gross = p["qty"] * float(r["exit_price"])
            sell_cost = gross * SELL_COST_BPS / 10000.0
            cash += gross - sell_cost
            closed.append(r.to_dict())

        # Then enter.
        for trade_id, r in entry_map.get(date, []):
            symbol = str(r["symbol"])

            if symbol in positions or len(positions) >= MAX_POSITIONS:
                continue

            try:
                open_raw = float(px.loc[(date, symbol), "open"])
            except Exception:
                continue

            if not np.isfinite(open_raw) or open_raw <= 0:
                continue

            current_equity = cash + sum(
                p["qty"] * p["last_close"] for p in positions.values()
            )

            target = min(
                current_equity * MAX_ALLOCATION_PER_POSITION,
                cash / (1 + BUY_COST_BPS / 10000.0)
            )

            if target <= 0:
                continue

            buy_px = apply_slippage(open_raw, "buy")
            cost = target * BUY_COST_BPS / 10000.0
            qty = target / buy_px

            cash -= target + cost

            positions[symbol] = {
                "qty": qty,
                "entry_price": buy_px,
                "last_close": buy_px,
            }

        # Mark.
        for symbol, p in positions.items():
            try:
                close_px = float(px.loc[(date, symbol), "close"])
                if np.isfinite(close_px):
                    p["last_close"] = close_px
            except Exception:
                pass

        position_value = sum(
            p["qty"] * p["last_close"]
            for p in positions.values()
        )

        eq_rows.append({
            "date": date,
            "equity": cash + position_value,
            "open_positions": len(positions),
        })

    return pd.DataFrame(eq_rows), pd.DataFrame(closed)

wf_port_rows = []

for fold, w in enumerate(wf_windows, 1):
    for regime in REGIMES:
        if nifty.empty and regime != "none":
            continue

        for stop in TRAILING_STOPS:
            trades = trade_sets[(regime, stop)]

            eq, closed = simulate_portfolio_subset(
                features,
                trades,
                w["test_start"],
                w["test_end"]
            )

            if eq.empty:
                continue

            m = portfolio_metrics(eq, closed)

            wf_port_rows.append({
                "fold": fold,
                "regime": regime,
                "trailing_stop": stop,
                **m
            })

wf_portfolio = pd.DataFrame(wf_port_rows)

if not wf_portfolio.empty:
    display(
        wf_portfolio[
            [
                "fold","regime","trailing_stop",
                "ending_equity","CAGR","max_drawdown",
                "Sharpe","Sortino","Calmar",
                "closed_trades","avg_open_positions"
            ]
        ].sort_values(["fold","regime","trailing_stop"])
    )

wf_portfolio.to_csv(
    RESULTS_DIR / "v3_walk_forward_portfolio_validation.csv",
    index=False
)


## 17. Walk-forward aggregate stability statistics


In [ ]:
if not wf_portfolio.empty:
    wf_summary = (
        wf_portfolio
        .groupby(["regime","trailing_stop"], as_index=False)
        .agg(
            folds=("fold","count"),
            mean_CAGR=("CAGR","mean"),
            median_CAGR=("CAGR","median"),
            mean_max_drawdown=("max_drawdown","mean"),
            worst_max_drawdown=("max_drawdown","min"),
            mean_Sharpe=("Sharpe","mean"),
            median_Sharpe=("Sharpe","median"),
            mean_Sortino=("Sortino","mean"),
            mean_Calmar=("Calmar","mean"),
            total_closed_trades=("closed_trades","sum"),
        )
    )

    display(
        wf_summary.sort_values(["regime","trailing_stop"])
    )

    wf_summary.to_csv(
        RESULTS_DIR / "v3_walk_forward_summary.csv",
        index=False
    )
else:
    wf_summary = pd.DataFrame()
    print("No walk-forward portfolio results.")


## 18. Causal implementation checks


In [ ]:
all_trade_sets = [
    x for x in trade_sets.values()
    if not x.empty
]

if all_trade_sets:
    all_trades = pd.concat(all_trade_sets, ignore_index=True)

    assert (
        pd.to_datetime(all_trades["entry_date"]) >
        pd.to_datetime(all_trades["signal_date"])
    ).all()

    assert (
        pd.to_datetime(all_trades["exit_date"]) >
        pd.to_datetime(all_trades["entry_date"])
    ).all()

    assert (all_trades["entry_price"] > 0).all()
    assert (all_trades["exit_price"] > 0).all()
    assert (all_trades["hold_days"] >= 1).all()

    print("✓ signal_date < entry_date")
    print("✓ entry_date < exit_date")
    print("✓ positive entry/exit prices")
    print("✓ positive holding periods")
else:
    print("No trades available for checks.")

print(
    "✓ Dataset freshness:",
    actual_max.date(),
    ">=",
    EXPECTED_MIN_END_DATE.date()
)


## 19. Save run manifest


In [ ]:
manifest = {
    "version": "V3",
    "data_dir": str(DATA_DIR),
    "results_dir": str(RESULTS_DIR),
    "actual_dataset_min_date": str(actual_min.date()),
    "actual_dataset_max_date": str(actual_max.date()),
    "expected_min_end_date": str(EXPECTED_MIN_END_DATE.date()),
    "starting_capital": STARTING_CAPITAL,
    "max_positions": MAX_POSITIONS,
    "entry_score_min": ENTRY_SCORE_MIN,
    "trailing_stops": TRAILING_STOPS,
    "hard_initial_stop": False,
    "failure_exit_days": FAILURE_EXIT_DAYS,
    "max_hold_days": MAX_HOLD_DAYS,
    "slippage_bps": SLIPPAGE_BPS,
    "buy_cost_bps": BUY_COST_BPS,
    "sell_cost_bps": SELL_COST_BPS,
    "regimes": REGIMES,
    "walk_forward_train_years": WF_TRAIN_YEARS,
    "walk_forward_validation_years": WF_VALID_YEARS,
    "walk_forward_test_years": WF_TEST_YEARS,
    "execution_model": "close_T_decision_to_open_T_plus_1_execution",
}

with open(RESULTS_DIR / "v3_run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(json.dumps(manifest, indent=2, default=str))


## 20. Final checklist

Before interpreting performance:

1. The freshness guard must pass.
2. Verify the actual dataset end date printed above.
3. Compare all six trailing stops.
4. Compare all available Nifty regimes.
5. Inspect calendar-year returns for the exact configuration.
6. Inspect walk-forward test results, not just full-sample results.
7. Validate corporate actions and price adjustments.
8. Add liquidity/ADV and participation-rate constraints before live use.
9. Add survivorship-bias controls / historical universe membership.
10. Add point-in-time fundamentals only after the technical pipeline is stable.

The notebook intentionally does **not** select or rank a preferred configuration automatically.
